# ST-DIF Reproduction Report

**Goal**: Reproduce Table 5 from the ST-DIF paper 

---

## Section 0: Sanity Checks

Before running any experiments, verify the environment and dependencies are correctly configured.

### 0.1 Working directory and Python environment

In [1]:
import os
import sys
import platform

# This notebook lives in examples/, so step up to repo root
os.chdir('..')
print(f"Working dir : {os.getcwd()}")
print(f"Python      : {platform.python_version()}")
print(f"Executable  : {sys.executable}")

Working dir : c:\Users\Ao\Desktop\crowd-framework-master\crowd-framework-master
Python      : 3.10.20
Executable  : c:\Users\Ao\.conda\envs\st_dif_env\python.exe


### 0.2 PyTorch and CUDA

In [2]:
import torch

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch     : 2.5.1+cu121
CUDA avail  : True
GPU name    : NVIDIA GeForce RTX 4070 SUPER
VRAM total  : 12.0 GB
Using device: cuda


### 0.3 PyTorch Geometric stack

In [3]:
import torch_geometric
import torch_geometric_temporal
from torch_scatter import scatter
from torch_sparse import SparseTensor

print(f"torch_geometric          : {torch_geometric.__version__}")
print(f"torch_geometric_temporal : {torch_geometric_temporal.__version__}")
print("torch_scatter, torch_sparse: imported OK")

c:\Users\Ao\.conda\envs\st_dif_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch_geometric          : 2.7.0
torch_geometric_temporal : 0.54.0
torch_scatter, torch_sparse: imported OK


### 0.4 `st_dif` package and the 5 models from `sten.py`

The notebook depends on Dongdong's updated `sten.py` (which adds `DenseGCNGRU`, `RecurrentGCN`, `DenseGCLSTM`, `DenseLRGCN`, `DenseMPNNLSTM`). Two helper files needed to be added before this import works:

- `evolvegcno.py` — re-exports `glorot` and `GCNConv_Fixed_W` from PyG / PyG-Temporal (was missing from the repo)
- `temporal_only.py` — provides `GRU_only` (was missing from the GitHub repo but present in the PyPI wheel; copied from there)

In [4]:
import st_dif

# Confirm st_dif is loaded from the editable repo, not site-packages
print(f"st_dif location: {st_dif.__file__}")

from st_dif.models.sten import (
    DenseGCNGRU,
    RecurrentGCN,
    DenseGCLSTM,
    DenseLRGCN,
    DenseMPNNLSTM,
)
print("All 5 models imported successfully:")
for cls in [DenseGCNGRU, RecurrentGCN, DenseGCLSTM, DenseLRGCN, DenseMPNNLSTM]:
    print(f"  - {cls.__name__}")

st_dif location: C:\Users\Ao\Desktop\crowd-framework-master\crowd-framework-master\src\st_dif\__init__.py
All 5 models imported successfully:
  - DenseGCNGRU
  - RecurrentGCN
  - DenseGCLSTM
  - DenseLRGCN
  - DenseMPNNLSTM


### 0.5 Data files

The GCS dataset should already be present in `data/gcs/gcs-processed/`.

In [5]:
from pathlib import Path

gcs_dir = Path('data/gcs/gcs-processed')
expected = ['flow_df_gcs.csv', 'GCS.cfg']

print(f"Looking in: {gcs_dir.resolve()}\n")
for fname in expected:
    fpath = gcs_dir / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f"  ✓ {fname:25s} ({size_kb:8.1f} KB)")
    else:
        print(f"  ✗ {fname:25s} MISSING")

Looking in: C:\Users\Ao\Desktop\crowd-framework-master\crowd-framework-master\data\gcs\gcs-processed

  ✓ flow_df_gcs.csv           (  2049.6 KB)
  ✓ GCS.cfg                   (     0.5 KB)


---

## Section 1: Load GCS dataset

Following the paper's Section 5: GCS dataset processed at 1.25 FPS, with PARs defined according to `data/gcs/gcs-processed/GCS.cfg`. The data loader returns a `StaticGraphTemporalSignal` object compatible with PyTorch Geometric Temporal.

### 1.1 Configuration (`Args`)

Hyperparameters following the paper's Section 5.2 — except `forecasting_horizon`, which we set per experiment. For this section we use **T=20** (paper's shortest horizon).

In [6]:
class Args:
    # Dataset
    DATASET = 'GCS'
    train_ratio = 0.7
    val_ratio = 0.0
    test_ratio = 0.3

    # Training (paper Section 5.2)
    batch_size = 32
    lr = 0.001
    epochs = 40

    # Forecasting (paper uses {20, 60, 120, 240}; start with shortest)
    forecasting_horizon = 20

    # Saving
    save_model = False
    save_dir = './checkpoints'

args = Args()

# Print summary
print("--- Args ---")
for k, v in vars(Args).items():
    if not k.startswith('_'):
        print(f"  {k:22s} = {v}")

--- Args ---
  DATASET                = GCS
  train_ratio            = 0.7
  val_ratio              = 0.0
  test_ratio             = 0.3
  batch_size             = 32
  lr                     = 0.001
  epochs                 = 40
  forecasting_horizon    = 20
  save_model             = False
  save_dir               = ./checkpoints


### 1.2 Load dataset and dataloaders

In [7]:
from st_dif.data_utils import get_pyg_temporal_dataset, get_loaders

dataset, _ = get_pyg_temporal_dataset(args.DATASET, args.forecasting_horizon)

train_loader, val_loader, test_loader = get_loaders(
    dataset,
    args.batch_size,
    args.train_ratio,
    args.val_ratio,
    args.test_ratio,
    device,
)

# Inspect
n_samples = len(list(dataset))
sample = next(iter(dataset))
print(f"Dataset type     : {type(dataset).__name__}")
print(f"Total samples    : {n_samples}")
print(f"Sample structure : {sample}")
print(f"  -> x  (node features over time): {tuple(sample.x.shape)}   # (N_PARs, in_channels, T)")
print(f"  -> y  (target horizon)         : {tuple(sample.y.shape)}    # (N_PARs, T)")
print(f"  -> edge_index                   : {tuple(sample.edge_index.shape)}  # (2, num_edges)")

c:\Users\Ao\Desktop\crowd-framework-master\crowd-framework-master
Dataset type:   <torch_geometric_temporal.signal.static_graph_temporal_signal.StaticGraphTemporalSignal object at 0x00000278981A6860>
Number of samples / sequences:  5173
Data(x=[9, 2, 20], edge_index=[2, 36], edge_attr=[36], y=[9, 20])
Number of train buckets:  3621
Number of val buckets:  0
Number of test buckets:  1552
Dataset type     : StaticGraphTemporalSignal
Total samples    : 5173
Sample structure : Data(x=[9, 2, 20], edge_index=[2, 36], edge_attr=[36], y=[9, 20])
  -> x  (node features over time): (9, 2, 20)   # (N_PARs, in_channels, T)
  -> y  (target horizon)         : (9, 20)    # (N_PARs, T)
  -> edge_index                   : (2, 36)  # (2, num_edges)


### 1.3 Static edge index

Since the GCS graph topology is time-invariant (the floor plan doesn't change), we extract `edge_index` once and reuse it across all timesteps.

In [8]:
# Static edge index (extract once)
for snapshot in dataset:
    static_edge_index = snapshot.edge_index.to(device)
    break

print(f"Edge index shape: {static_edge_index.shape}")
print(f"Edge index:\n{static_edge_index}")

Edge index shape: torch.Size([2, 36])
Edge index:
tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5,
         5, 5, 6, 6, 6, 6, 7, 7, 7, 7, 8, 8],
        [1, 2, 3, 0, 2, 3, 0, 1, 3, 4, 5, 0, 1, 2, 4, 5, 2, 3, 5, 6, 7, 2, 3, 4,
         6, 7, 4, 5, 7, 8, 4, 5, 6, 8, 6, 7]], device='cuda:0')


---

## Section 0–1 Summary

This section records what was verified before any model was trained. Sections 2–4 will report what was actually observed when training and evaluating, with no presumption that observations will match the paper.

### Environment (verified above)

| Item | Source of constraint | Observed |
|---|---|---|
| Python | 3.10 (repo README) | 3.10.20 |
| PyTorch | 2.5.1+cu121 (repo README) | 2.5.1+cu121 |
| CUDA available | — | True |
| GPU | — | RTX 4070 SUPER, 12 GB |
| `torch_geometric` | required (repo README) | 2.7.0 |
| `torch_geometric_temporal` | required (repo README) | 0.54.0 |
| `torch_scatter`, `torch_sparse` | required (repo README) | imported without error |

### `st_dif` package (verified above)

| Item | Source of constraint | Observed |
|---|---|---|
| Install mode | editable (`pip install -e .`, repo README) | resolved to repo `src/` |
| Models needed for Table 5 | Paper §5.6 lists `RecurrentGCN (DCRNN)`, `GC-LSTM`, `LRGCN`, `MPNN-LSTM`, `Dense-GCN-GRU` | `DenseGCNGRU`, `RecurrentGCN`, `DenseGCLSTM`, `DenseLRGCN`, `DenseMPNNLSTM` all imported |

**Files that had to be added before this notebook could import `st_dif.models`** (issues encountered, not yet reported to the authors):

- `src/st_dif/models/evolvegcno.py` was not present in the GitHub repo (master, Dat-branch, semester-code-clean), in any commit of the git history, or in the published `st_dif==0.1.2` PyPI wheel. The new `sten.py` line 16 imports `glorot` and `GCNConv_Fixed_W` from this module. Both names exist upstream (`torch_geometric.nn.inits.glorot`, `torch_geometric_temporal.nn.recurrent.evolvegcno.GCNConv_Fixed_W`); a re-export stub was created at the expected path. The patch is documented for review — it has not been confirmed with the authors that this is what their local `evolvegcno.py` contains.
- `src/st_dif/models/temporal_only.py` was missing from the GitHub repo but present in the `st_dif==0.1.2` PyPI wheel. The file was copied verbatim from the wheel.

These two patches are necessary preconditions for running anything; the reproduction below is conditional on whether they faithfully reflect the original code.

### GCS dataset (loaded above)

| Item | Paper / source | Observed |
|---|---|---|
| `flow_df_gcs.csv` in `data/gcs/gcs-processed/` | required by loader | present, 2049.6 KB |
| `GCS.cfg` in same directory | required by loader | present, 0.5 KB |
| Number of PARs (nodes) | 9 (Paper Fig. 2(a)) | 9 |
| Input feature dim | D = 2 (Paper §4.1, item 4) | 2 |
| Number of edges in `edge_index` | not specified in paper | 36 directed entries (= 18 undirected, if assumed symmetric) |
| Train / Val / Test split | not specified in paper | 70 / 0 / 30 % (3621 / 0 / 1552), following Vivian's `demo_forecast_crowd_flow.ipynb` |
| Total sequences | not specified in paper | 5173 |

### Training hyperparameters (set in `Args`, not yet executed)

| Item | Paper §5.2 | Args value in this notebook |
|---|---|---|
| Epochs | 40 | 40 |
| Batch size | 32 | 32 |
| Learning rate | 0.001 | 0.001 |
| Optimizer | Adam | (to be confirmed by inspecting `train()`) |
| Forecasting horizons | {20, 60, 120, 240} | will iterate over the same set in Section 3 |
| Independent runs per setting | 5 | will use 5 in Section 3 |
| Reported statistic | "mean and min-max range" (i.e., (max − min) / 2) | will compute the same in Section 3 |

### What is *not* claimed by this section

- That the training results will reproduce Table 5. That is the question Section 4 will answer empirically.
- That the patched `evolvegcno.py` is identical to the authors' original. It is a functional re-export and may differ from their local file.
- That items marked "not specified in paper" are correct — they are inherited from the repo and demo and are recorded only for transparency.



---

## Section 2: Smoke test

A 3-epoch run of `DenseGCNGRU` at T=20 to confirm the train/evaluate pipeline executes end-to-end. The output number is for pipeline validation only — it is not compared against the paper. Section 3 will run the actual reproduction grid.

In [9]:
from st_dif.train_test_utils import train, evaluate

# Build model
smoke_model = DenseGCNGRU(
    in_channels=2,
    periods=args.forecasting_horizon,
    batch_size=args.batch_size,
).to(device)
print(smoke_model)
print()

# Train for 3 epochs (not 40)
smoke_model, smoke_ckpt = train(
    smoke_model,
    train_loader,
    val_loader,
    static_edge_index,
    num_epochs=3,
    lr=args.lr,
)

# Evaluate
smoke_model, smoke_ckpt = evaluate(
    smoke_model,
    test_loader,
    static_edge_index,
    checkpoint_dict=smoke_ckpt,
)

print(f"\nSmoke test finished.")
print(f"  Test MSE: {smoke_ckpt['test_mse']:.4f}")
print(f"  Test MAE: {smoke_ckpt['test_mae']:.4f}")
print("  (These numbers are pipeline-validation only, not a reproduction claim.)")

DenseGCNGRU(
  (densegcn): DeepGCNLayer(block=dense)
  (gru): GRU(130, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=20, bias=True)
)

Epoch 0 Step 0 train MSE: 1.1141
Epoch 0 Step 100 train MSE: 0.2652
Epoch 0 Average Training MSE: 0.2499
Epoch 1 Step 0 train MSE: 0.1925
Epoch 1 Step 100 train MSE: 0.1182
Epoch 1 Average Training MSE: 0.1160
Epoch 2 Step 0 train MSE: 0.1085
Epoch 2 Step 100 train MSE: 0.1079
Epoch 2 Average Training MSE: 0.1079
Test MSE: 0.1067
Test MAE: 0.2276

Smoke test finished.
  Test MSE: 0.1067
  Test MAE: 0.2276
  (These numbers are pipeline-validation only, not a reproduction claim.)


---

## Section 3: Full reproduction grid

Goal: produce the GCS column of paper Table 5 — MSE and MAE for 5 models × 4 horizons × 5 independent runs at 40 epochs each (= 100 runs total).

Settings follow paper §5.2: `epochs=40`, `batch_size=32`, `lr=0.001`, Adam optimizer (as called by `train()`).

Each run uses a different random seed (`torch.manual_seed(run_idx)`) for reproducibility of this notebook itself.

Results are written to `forecasting_horizon_results.csv` incrementally — if the kernel dies mid-run, partial results are preserved.


In [15]:
import csv
import time
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from st_dif.train_test_utils import train, evaluate

# ============ Config ============
forecasting_horizons = [20, 60, 120, 240]
model_names          = ['DenseGCNGRU', 'RecurrentGCN', 'DenseGCLSTM', 'DenseLRGCN', 'DenseMPNNLSTM']
num_runs             = 5
num_epochs           = 40

# Hidden size for DenseGCLSTM (not specified in paper §5.2; choosing 64 to match D_GRU)
GCLSTM_HIDDEN = 64

# Portable path (fork edit): results live under the repo root.
# Original run used a local Windows path (C:\Users\Ao\...\results_run2).
results_dir = Path('results_reproduction')
results_dir.mkdir(exist_ok=True)
results_csv = results_dir / 'forecasting_horizon_results.csv'

# ============ Model factory (per-model signature) ============
def build_model(name, horizon, batch_size):
    if name == 'DenseGCNGRU':
        return DenseGCNGRU(in_channels=2, periods=horizon, batch_size=batch_size)
    if name == 'RecurrentGCN':
        return RecurrentGCN(in_channels=2, periods=horizon, batch_size=batch_size)
    if name == 'DenseGCLSTM':
        return DenseGCLSTM(in_channels=2, hidden_channels=GCLSTM_HIDDEN, periods=horizon)
    if name == 'DenseLRGCN':
        return DenseLRGCN(in_channels=2, periods=horizon, batch_size=batch_size)
    if name == 'DenseMPNNLSTM':
        return DenseMPNNLSTM(in_channels=2, periods=horizon)
    raise ValueError(f'Unknown model: {name}')

# ============ Resume support: see what's already done ============
done = set()
if results_csv.exists():
    df_done = pd.read_csv(results_csv)
    for _, r in df_done.iterrows():
        done.add((int(r['horizon']), str(r['model']), int(r['run_idx'])))
    print(f"Found existing CSV with {len(done)} completed runs. Will skip them.")
else:
    with results_csv.open('w', newline='') as f:
        csv.writer(f).writerow(
            ['horizon', 'model', 'run_idx', 'test_mse', 'test_mae', 'wall_seconds']
        )
    print(f"Created new CSV at: {results_csv}")

# ============ Main grid ============
total_runs = len(forecasting_horizons) * len(model_names) * num_runs
already_done = len(done)
remaining = total_runs - already_done
print(f"Total runs: {total_runs}, already done: {already_done}, remaining: {remaining}\n")

run_counter = 0          # global index, including done ones
done_this_session = 0    # only count newly run
overall_start = time.time()

for horizon in forecasting_horizons:
    # Reload data once per horizon
    dataset, _ = get_pyg_temporal_dataset(args.DATASET, horizon)
    train_loader, val_loader, test_loader = get_loaders(
        dataset, args.batch_size,
        args.train_ratio, args.val_ratio, args.test_ratio,
        device,
    )
    for snapshot in dataset:
        static_edge_index = snapshot.edge_index.to(device)
        break

    for model_name in model_names:
        for run_idx in range(num_runs):
            run_counter += 1
            key = (horizon, model_name, run_idx)
            if key in done:
                continue   # skip already-done

            torch.manual_seed(run_idx)
            np.random.seed(run_idx)

            t0 = time.time()
            try:
                model = build_model(model_name, horizon, args.batch_size).to(device)
                model, ckpt = train(
                    model, train_loader, val_loader, static_edge_index,
                    num_epochs=num_epochs, lr=args.lr,
                )
                model, ckpt = evaluate(
                    model, test_loader, static_edge_index, checkpoint_dict=ckpt
                )
                mse = float(ckpt['test_mse'])
                mae = float(ckpt['test_mae'])
                status = 'ok'
            except Exception as e:
                mse = float('nan')
                mae = float('nan')
                status = f'ERROR: {type(e).__name__}: {e}'
                print(f'  !!! {model_name} run={run_idx} failed: {status}')
            finally:
                # Free GPU memory before next run
                try:
                    del model
                except NameError:
                    pass
                torch.cuda.empty_cache()

            elapsed = time.time() - t0

            with results_csv.open('a', newline='') as f:
                csv.writer(f).writerow(
                    [horizon, model_name, run_idx, mse, mae, f'{elapsed:.1f}']
                )

            done_this_session += 1
            session_elapsed = time.time() - overall_start
            avg_per_run = session_elapsed / done_this_session
            eta_sec = avg_per_run * (remaining - done_this_session)

            print(
                f'[{run_counter:3d}/{total_runs}] '
                f'T={horizon:3d}  {model_name:14s} run={run_idx}  '
                f'MSE={mse:.4f}  MAE={mae:.4f}  '
                f'({elapsed:5.1f}s | session {session_elapsed/60:5.1f} min | '
                f'ETA {eta_sec/3600:.1f} h)'
            )

print(f'\nGrid complete. Session time: {(time.time() - overall_start)/60:.1f} min')
print(f'Results: {results_csv.resolve()}')

Created new CSV at: C:\Users\Ao\Desktop\crowd-framework-master\results_run2\forecasting_horizon_results.csv
Total runs: 100, already done: 0, remaining: 100

c:\Users\Ao\Desktop\crowd-framework-master\crowd-framework-master
Dataset type:   <torch_geometric_temporal.signal.static_graph_temporal_signal.StaticGraphTemporalSignal object at 0x000002793AE507F0>
Number of samples / sequences:  5173
Data(x=[9, 2, 20], edge_index=[2, 36], edge_attr=[36], y=[9, 20])
Number of train buckets:  3621
Number of val buckets:  0
Number of test buckets:  1552
Epoch 0 Step 0 train MSE: 1.1136
Epoch 0 Step 100 train MSE: 0.2917
Epoch 0 Average Training MSE: 0.2724
Epoch 1 Step 0 train MSE: 0.1414
Epoch 1 Step 100 train MSE: 0.1187
Epoch 1 Average Training MSE: 0.1186
Epoch 2 Step 0 train MSE: 0.1364
Epoch 2 Step 100 train MSE: 0.1083
Epoch 2 Average Training MSE: 0.1085
Epoch 3 Step 0 train MSE: 0.1046
Epoch 3 Step 100 train MSE: 0.1047
Epoch 3 Average Training MSE: 0.1051
Epoch 4 Step 0 train MSE: 0.1253

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

# Portable path (fork edit): results live under the repo root.
# Original run used a local Windows path (C:\Users\Ao\...\results_run2).
results_dir = Path('results_reproduction')
results_dir.mkdir(exist_ok=True)

results_csv = results_dir / 'forecasting_horizon_results.csv'
print(f"New CSV path: {results_csv}")
print(f"Exists already: {results_csv.exists()}")

New CSV path: C:\Users\Ao\Desktop\crowd-framework-master\results_run2\forecasting_horizon_results.csv
Exists already: False


In [13]:
import inspect

print("=== Signature of each model's __init__ ===\n")
for name, cls in [
    ('DenseGCNGRU',   DenseGCNGRU),
    ('RecurrentGCN',  RecurrentGCN),
    ('DenseGCLSTM',   DenseGCLSTM),
    ('DenseLRGCN',    DenseLRGCN),
    ('DenseMPNNLSTM', DenseMPNNLSTM),
]:
    sig = inspect.signature(cls.__init__)
    params = list(sig.parameters.values())[1:]
    print(f"{name}:")
    for p in params:
        default = '' if p.default is inspect.Parameter.empty else f' = {p.default!r}'
        print(f"  {p.name}{default}")
    print()

=== Signature of each model's __init__ ===

DenseGCNGRU:
  in_channels
  periods
  batch_size
  improved = False
  cached = False
  add_self_loops = True

RecurrentGCN:
  in_channels
  periods
  batch_size
  hidden_gcn = 128
  hidden_rnn = 64

DenseGCLSTM:
  in_channels
  hidden_channels
  periods
  K = 3
  normalization = 'sym'
  bias = True

DenseLRGCN:
  in_channels
  periods
  batch_size
  num_relations = 1
  num_bases = 1

DenseMPNNLSTM:
  in_channels
  periods
  hidden_size = 128
  dropout = 0.0



In [14]:
import inspect

print("=== DenseGCLSTM.__init__ source ===")
print(inspect.getsource(DenseGCLSTM.__init__))
print()
print("=== DenseLRGCN.__init__ source ===")
print(inspect.getsource(DenseLRGCN.__init__))
print()
print("=== DenseMPNNLSTM.__init__ source ===")
print(inspect.getsource(DenseMPNNLSTM.__init__))

=== DenseGCLSTM.__init__ source ===
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        periods: int,
        K: int = 3,
        normalization: str = "sym",
        bias: bool = True,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.periods = periods

        self.cell = GCLSTM(
            in_channels=in_channels,
            out_channels=hidden_channels,
            K=K,
            normalization=normalization,
            bias=bias,
        )

        self.fc = torch.nn.Linear(hidden_channels, periods)


=== DenseLRGCN.__init__ source ===
    def __init__(
        self,
        in_channels: int,
        periods: int,
        batch_size: int,
        num_relations: int = 1,
        num_bases: int = 1,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.periods = periods
        self.batch_size = batch_size
        self.nu

---

## Section 4: Diagnostic findings and patched `DenseGCNGRU`

Goal: prepare the model code needed to reproduce paper Tables 3 (sensitivity to GCN depth $K$) and 4 (sensitivity to model size). Both tables vary internal hyperparameters of `DenseGCNGRU`. Two issues in the current `sten.py` need to be addressed before either table can be reproduced cleanly. The two issues are documented in §4.1; §4.2 introduces patched classes that fix them while preserving the original behavior under default arguments; §4.3 verifies that equivalence numerically.


### 4.1 Audit of `DenseGCNGRU` and `KLayerGCNConv`

**Issue 1 — `KLayerGCNConv.forward` returns after the first GCN layer regardless of `K`.**

`sten.py` lines 507–510:

```python
def forward(self, x, edge_index, edge_weight):
    for k in range(len(self.convs)):
        x = self.convs[k].forward(x, edge_index, edge_weight)
        return x   # <- inside the for loop
```

The `return x` is one indent level too deep, so the function exits after `k = 0`. For `K = 2` or `K = 3`, the additional `GCNConv` layers exist as `nn.Module` parameters but are never called in the forward pass. As a consequence, paper Table 3 — which varies $K \in \{1, 2, 3\}$ — does not actually test GCN depth: all three configurations execute the same single-hop forward pass with different amounts of dead weight.

**Issue 2 — `K`, `D_{GCN}`, and `D_{GRU}` are hardcoded inside `_setup_layers`.**

`sten.py` lines 538–550:

```python
def _setup_layers(self):
    self.densegcn = DeepGCNLayer(
        conv=KLayerGCNConv(
            K=3,                # hardcoded
            in_channels=self.in_channels,
            out_channels=128,   # hardcoded = D_GCN
            ...
        ),
        ...
    )
    self.gru = torch.nn.GRU(130, 64, 2, batch_first=True)
    # 130 = D_GCN + in_channels (dense concat)
    # 64  = D_GRU
    # 2   = num GRU layers
```

The constructor `__init__` does not expose `K`, `D_{GCN}`, or `D_{GRU}` as arguments. Reproducing Tables 3 and 4 from the original code therefore requires editing `sten.py` between runs, with no audit trail of which numbers were edited for which row.

**Action.** §4.2 defines patched classes `KLayerGCNConvPatched` and `DenseGCNGRUPatched` that:

- Expose `K`, `D_GCN`, `D_GRU`, `num_gru_layers` as constructor arguments. Defaults preserve the original hardcoded values (`K=3`, `D_GCN=128`, `D_GRU=64`, `num_gru_layers=2`).
- Accept a `fix_forward_bug` flag — `False` matches the published forward (returns after first layer); `True` applies all `K` GCN layers.
- Are otherwise structurally identical to the originals, so that with default arguments and `fix_forward_bug=False`, the patched class produces a bit-identical `state_dict` and forward output to the original. §4.3 verifies this numerically.

The patch is inline in this notebook only — `sten.py` itself is not modified.


### 4.2 Patched class definitions


In [ ]:
import torch
from torch_geometric.nn import GCNConv
from torch_geometric.nn.models import DeepGCNLayer


class KLayerGCNConvPatched(torch.nn.Module):
    """Patched copy of `st_dif.models.sten.KLayerGCNConv`.

    Adds `fix_forward_bug`:
      - False (default): replicate the original forward, which returns after
        the first GCN layer regardless of K.
      - True : apply all K GCN layers in sequence.

    Module construction is otherwise identical to the original, so under the
    same RNG state the parameter tensors match bit-for-bit.
    """

    def __init__(
        self,
        K: int,
        in_channels: int,
        out_channels: int,
        node_dim: int,
        improved: bool = True,
        cached: bool = False,
        add_self_loops: bool = True,
        fix_forward_bug: bool = False,
    ):
        super().__init__()
        self.fix_forward_bug = fix_forward_bug
        convs = []
        for k in range(K):
            if k == 0:
                convs.append(GCNConv(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    node_dim=node_dim,
                    improved=improved,
                    cached=cached,
                    add_self_loops=add_self_loops,
                ))
            else:
                convs.append(GCNConv(
                    in_channels=out_channels,
                    out_channels=out_channels,
                    node_dim=node_dim,
                    improved=improved,
                    cached=cached,
                    add_self_loops=add_self_loops,
                ))
        self.convs = torch.nn.Sequential(*convs)

    def forward(self, x, edge_index, edge_weight):
        for k in range(len(self.convs)):
            x = self.convs[k].forward(x, edge_index, edge_weight)
            if not self.fix_forward_bug:
                return x          # paper-faithful: return after first layer
        return x                  # fixed: applied all K layers


class DenseGCNGRUPatched(torch.nn.Module):
    """Patched copy of `st_dif.models.sten.DenseGCNGRU`.

    Exposes `K`, `D_GCN`, `D_GRU`, `num_gru_layers`, and `fix_forward_bug`.
    Defaults reproduce the hardcoded values in the original
    (K=3, D_GCN=128, D_GRU=64, num_gru_layers=2, fix_forward_bug=False).
    """

    def __init__(
        self,
        in_channels: int,
        periods: int,
        batch_size: int,
        K: int = 3,
        D_GCN: int = 128,
        D_GRU: int = 64,
        num_gru_layers: int = 2,
        fix_forward_bug: bool = False,
        improved: bool = False,
        cached: bool = False,
        add_self_loops: bool = True,
    ):
        super().__init__()
        self.in_channels    = in_channels
        self.periods        = periods
        self.batch_size     = batch_size
        self.K              = K
        self.D_GCN          = D_GCN
        self.D_GRU          = D_GRU
        self.num_gru_layers = num_gru_layers
        self.fix_forward_bug = fix_forward_bug
        self.improved        = improved
        self.cached          = cached
        self.add_self_loops  = add_self_loops
        self._setup_layers()

    def _setup_layers(self):
        self.densegcn = DeepGCNLayer(
            conv=KLayerGCNConvPatched(
                K=self.K,
                in_channels=self.in_channels,
                out_channels=self.D_GCN,
                node_dim=1,
                improved=self.improved,
                cached=self.cached,
                add_self_loops=self.add_self_loops,
                fix_forward_bug=self.fix_forward_bug,
            ),
            norm=None,
            act=None,
            dropout=0,
            block='dense',
        )
        gru_input = self.D_GCN + self.in_channels   # dense-concat with original features
        self.gru = torch.nn.GRU(gru_input, self.D_GRU, self.num_gru_layers, batch_first=True)
        self.fc  = torch.nn.Linear(self.D_GRU, self.periods)

    def forward(self, X, edge_index, edge_weight=None):
        gru_in = torch.zeros(
            X.shape[0], X.shape[1], X.shape[3],
            self.D_GCN + self.in_channels,
        ).to(X.device)
        for t in range(X.shape[3]):
            gcn_out = self.densegcn(X[:, :, :, t], edge_index, edge_weight)
            gru_in[:, :, t, :] = gcn_out
        gru_in = gru_in.flatten(start_dim=0, end_dim=1)
        gru_out, _ = self.gru(gru_in)
        out = self.fc(gru_out[:, -1, :])
        out = out.view(X.shape[0], X.shape[1], self.periods, -1)
        return out.squeeze(dim=3)


print("Patched classes defined: KLayerGCNConvPatched, DenseGCNGRUPatched")


### 4.3 Equivalence check: `DenseGCNGRUPatched(default args, fix_forward_bug=False)` ≡ `DenseGCNGRU`

If the patched class is constructed with default arguments and `fix_forward_bug=False`, it should be a drop-in replacement for the original. Under the same RNG seed two checks must pass:

1. `state_dict()` keys and tensor values match exactly.
2. The forward output on the same input is bit-identical (or within float-precision tolerance).

If either fails, the patch has unintentionally diverged from the original, and any sensitivity numbers derived from it would not be comparable to the paper's. The cell below asserts both.


In [ ]:
# Reload GCS at T=20 so the equivalence check uses the right edge_index
# (Section 3 may have left static_edge_index pointing at T=240's loaders)
dataset_t20, _ = get_pyg_temporal_dataset('GCS', 20)
for snapshot in dataset_t20:
    static_edge_index_t20 = snapshot.edge_index.to(device)
    break

# 1. Same RNG seed -> identical state_dict
torch.manual_seed(0)
m_orig = DenseGCNGRU(in_channels=2, periods=20, batch_size=32).to(device)

torch.manual_seed(0)
m_patched = DenseGCNGRUPatched(in_channels=2, periods=20, batch_size=32).to(device)

sd_orig    = m_orig.state_dict()
sd_patched = m_patched.state_dict()

assert set(sd_orig.keys()) == set(sd_patched.keys()), (
    f"Different keys:\n  orig only: {set(sd_orig) - set(sd_patched)}\n"
    f"  patched only: {set(sd_patched) - set(sd_orig)}"
)
for k in sd_orig:
    assert torch.equal(sd_orig[k], sd_patched[k]), f"Tensor mismatch at {k}"
print(f"  ok  state_dict match: {len(sd_orig)} tensors, identical")

# 2. Same forward output on the same input
N_PARS = 9    # GCS has 9 PARs
B      = 4
T_in   = 20
torch.manual_seed(42)
x = torch.randn(B, N_PARS, 2, T_in, device=device)

m_orig.eval()
m_patched.eval()
with torch.no_grad():
    out_orig    = m_orig(x, static_edge_index_t20)
    out_patched = m_patched(x, static_edge_index_t20)

max_abs_diff = (out_orig - out_patched).abs().max().item()
print(f"  ok  forward output match: max |diff| = {max_abs_diff:.2e}")

assert torch.allclose(out_orig, out_patched, atol=1e-7), (
    f"Output mismatch! max abs diff = {max_abs_diff}"
)

print()
print("Equivalence verified. DenseGCNGRUPatched(defaults, fix_forward_bug=False) is")
print("a drop-in replacement for the original DenseGCNGRU. Safe to use as the base")
print("for Tables 3 (Section 5) and 4 (Section 6).")


---

## Section 5: Table 3 — sensitivity to GCN depth $K$

Paper Table 3 reports MSE/MAE for `DenseGCNGRU` on GCS at $T=20$ with $K \in \{1, 2, 3\}$, averaged over 5 runs. Given the `KLayerGCNConv.forward` bug documented in §4.1, the published numbers reflect a forward pass that always exits after the first GCN layer regardless of $K$. This section runs both modes:

- **paper_faithful** (`fix_forward_bug=False`) — reproduces the published behavior. Numbers should fall within Dongdong's reported min-max ranges if the rest of his configuration matches this notebook's.
- **fixed** (`fix_forward_bug=True`) — applies all $K$ GCN layers. Tests whether the K-sensitivity claim in paper §5.5.1 actually holds when $K$ is being respected.

Note on what each mode is testing:

| Mode | What varies with $K$ | What stays constant with $K$ |
|---|---|---|
| paper_faithful | total parameter count (more $K$ → more dead weight) | forward computation (always 1 GCN layer) |
| fixed | total parameter count *and* receptive field ($K$-hop) | nothing — both effects compound |

All other settings match paper §5.2 (`epochs=40`, `batch_size=32`, `lr=0.001`, Adam, $T=20$, 5 runs, GCS, 70/0/30 split).


### 5.1 Sweep configuration


In [ ]:
# Table 3 sweep config
T20             = 20
K_VALUES        = [1, 2, 3]
MODES_T3        = ['paper_faithful', 'fixed']   # paper_faithful first
NUM_RUNS_T3     = 5
NUM_EPOCHS_T3   = 40

# Held fixed across the K sweep (paper Table 3 only varies K)
D_GCN_FIXED     = 128
D_GRU_FIXED     = 64
NUM_GRU_LAYERS  = 2

# Portable path (fork edit): results live under the repo root.
# Original run used a local Windows path (C:\Users\Ao\...\results_run2).
results_dir = Path('results_reproduction')
results_dir.mkdir(exist_ok=True)
table3_csv  = results_dir / 'table3_K_sensitivity.csv'

n_total_t3 = len(MODES_T3) * len(K_VALUES) * NUM_RUNS_T3
print(f"Sweep size : |modes|={len(MODES_T3)} x |K|={len(K_VALUES)} x runs={NUM_RUNS_T3} = {n_total_t3} runs")
print(f"Horizon    : T = {T20}")
print(f"Output CSV : {table3_csv}")


### 5.2 Run sweep

Same incremental-CSV / resume pattern as Section 3 — partial results are preserved if the kernel dies.


In [ ]:
# Loaders at T=20 (this section never changes horizon)
dataset, _ = get_pyg_temporal_dataset(args.DATASET, T20)
train_loader, val_loader, test_loader = get_loaders(
    dataset, args.batch_size,
    args.train_ratio, args.val_ratio, args.test_ratio,
    device,
)
for snapshot in dataset:
    static_edge_index = snapshot.edge_index.to(device)
    break

# Resume support
done_t3 = set()
if table3_csv.exists():
    df_done = pd.read_csv(table3_csv)
    for _, r in df_done.iterrows():
        done_t3.add((str(r['mode']), int(r['K']), int(r['run_idx'])))
    print(f"Found existing CSV with {len(done_t3)} completed runs; will skip them.\n")
else:
    with table3_csv.open('w', newline='') as f:
        csv.writer(f).writerow(
            ['mode', 'K', 'D_GCN', 'D_GRU', 'run_idx', 'test_mse', 'test_mae', 'wall_seconds']
        )
    print(f"Created new CSV: {table3_csv}\n")

done_session = 0
overall_start = time.time()
counter = 0

for mode in MODES_T3:
    fix_flag = (mode == 'fixed')
    for K in K_VALUES:
        for run_idx in range(NUM_RUNS_T3):
            counter += 1
            key = (mode, K, run_idx)
            if key in done_t3:
                continue

            torch.manual_seed(run_idx)
            np.random.seed(run_idx)

            t0 = time.time()
            try:
                model = DenseGCNGRUPatched(
                    in_channels=2,
                    periods=T20,
                    batch_size=args.batch_size,
                    K=K,
                    D_GCN=D_GCN_FIXED,
                    D_GRU=D_GRU_FIXED,
                    num_gru_layers=NUM_GRU_LAYERS,
                    fix_forward_bug=fix_flag,
                ).to(device)

                model, ckpt = train(
                    model, train_loader, val_loader, static_edge_index,
                    num_epochs=NUM_EPOCHS_T3, lr=args.lr,
                )
                model, ckpt = evaluate(
                    model, test_loader, static_edge_index, checkpoint_dict=ckpt,
                )
                mse = float(ckpt['test_mse'])
                mae = float(ckpt['test_mae'])
            except Exception as e:
                mse = float('nan')
                mae = float('nan')
                print(f'  !!! mode={mode} K={K} run={run_idx} failed: {type(e).__name__}: {e}')
            finally:
                try:
                    del model
                except NameError:
                    pass
                torch.cuda.empty_cache()

            elapsed = time.time() - t0
            with table3_csv.open('a', newline='') as f:
                csv.writer(f).writerow(
                    [mode, K, D_GCN_FIXED, D_GRU_FIXED, run_idx, mse, mae, f'{elapsed:.1f}']
                )

            done_session += 1
            session_el = time.time() - overall_start
            avg = session_el / done_session
            eta = avg * (n_total_t3 - len(done_t3) - done_session)

            print(
                f'[{counter:2d}/{n_total_t3}] mode={mode:14s} K={K} run={run_idx}  '
                f'MSE={mse:.4f}  MAE={mae:.4f}  '
                f'({elapsed:5.1f}s | ETA {eta/60:5.1f} min)'
            )

print(f"\nSweep complete. Session time: {(time.time()-overall_start)/60:.1f} min")
print(f"Results: {table3_csv.resolve()}")


### 5.3 Aggregate and compare to paper Table 3

Reported statistic matches the paper: `mean ± (max − min) / 2` over 5 independent runs.


In [ ]:
df_t3 = pd.read_csv(table3_csv).dropna(subset=['test_mse', 'test_mae'])

def half_range(s):
    return (s.max() - s.min()) / 2

agg_t3 = (df_t3
    .groupby(['mode', 'K'])
    .agg(
        mse_mean = ('test_mse', 'mean'),
        mse_hr   = ('test_mse', half_range),
        mae_mean = ('test_mae', 'mean'),
        mae_hr   = ('test_mae', half_range),
        n_runs   = ('test_mse', 'count'),
    )
    .reset_index()
)

# Paper Table 3 reference numbers (from paper §5.5.1)
paper_t3 = pd.DataFrame([
    {'K': 1, 'paper_mse': '0.0905 ± 0.0009', 'paper_mae': '0.2089 ± 0.0012'},
    {'K': 2, 'paper_mse': '0.0894 ± 0.0010', 'paper_mae': '0.2080 ± 0.0010'},
    {'K': 3, 'paper_mse': '0.0891 ± 0.0009', 'paper_mae': '0.2075 ± 0.0013'},
])

print("=== Table 3 reproduction ===\n")
for mode in MODES_T3:
    sub = agg_t3[agg_t3['mode'] == mode]
    if len(sub) == 0:
        print(f"--- mode = {mode} : no runs yet ---\n")
        continue
    print(f"--- mode = {mode} ---")
    fmt = sub.merge(paper_t3, on='K')
    fmt['my_mse'] = fmt.apply(lambda r: f"{r['mse_mean']:.4f} ± {r['mse_hr']:.4f}", axis=1)
    fmt['my_mae'] = fmt.apply(lambda r: f"{r['mae_mean']:.4f} ± {r['mae_hr']:.4f}", axis=1)
    print(fmt[['K', 'paper_mse', 'my_mse', 'paper_mae', 'my_mae', 'n_runs']]
          .to_string(index=False))
    print()


---

## Section 6: Table 4 — sensitivity to model size

Paper Table 4 reports MSE/MAE for `DenseGCNGRU` on GCS at $T=20$ across three size configurations:

| Config | $D_{GCN}$ | $D_{GRU}$ |
|---|---|---|
| Standard | 128 | 64 |
| Medium | 64 | 32 |
| Tiny | 32 | 16 |

All with $K=3$ and 5 runs. Both $D_{GCN}$ and $D_{GRU}$ halve together at each step — Table 4 is a *combined* model-size analysis, not a single-variable sensitivity. The patched class allows them to be set independently in case a finer-grained sweep is needed later.

We run both modes here for symmetry with §5:

- **paper_faithful** — reproduces the published numbers (forward bug retained).
- **fixed** — runs all three sizes with proper $K=3$ depth. Informative as follow-up to §5.

The Standard / paper_faithful row of this section should agree with the $K=3$ / paper_faithful row of §5 within seed variance, since they use the same configuration. This serves as an internal consistency check.


### 6.1 Sweep configuration


In [ ]:
# Table 4 sweep config
SIZE_CONFIGS = [
    {'name': 'Standard', 'D_GCN': 128, 'D_GRU': 64},
    {'name': 'Medium',   'D_GCN':  64, 'D_GRU': 32},
    {'name': 'Tiny',     'D_GCN':  32, 'D_GRU': 16},
]
MODES_T4      = ['paper_faithful', 'fixed']
K_FIXED       = 3
NUM_RUNS_T4   = 5
NUM_EPOCHS_T4 = 40

table4_csv = results_dir / 'table4_size_sensitivity.csv'

n_total_t4 = len(MODES_T4) * len(SIZE_CONFIGS) * NUM_RUNS_T4
print(f"Sweep size : |modes|={len(MODES_T4)} x |sizes|={len(SIZE_CONFIGS)} x runs={NUM_RUNS_T4} = {n_total_t4} runs")
print(f"K fixed at : {K_FIXED}, horizon T = {T20}")
print(f"Output CSV : {table4_csv}")


### 6.2 Run sweep


In [ ]:
# Loaders at T=20 already in scope from §5.2 — reuse them

done_t4 = set()
if table4_csv.exists():
    df_done = pd.read_csv(table4_csv)
    for _, r in df_done.iterrows():
        done_t4.add((str(r['mode']), str(r['size_name']), int(r['run_idx'])))
    print(f"Found existing CSV with {len(done_t4)} completed runs; will skip them.\n")
else:
    with table4_csv.open('w', newline='') as f:
        csv.writer(f).writerow(
            ['mode', 'size_name', 'D_GCN', 'D_GRU', 'K', 'run_idx',
             'test_mse', 'test_mae', 'wall_seconds']
        )
    print(f"Created new CSV: {table4_csv}\n")

done_session = 0
ov_start = time.time()
ctr = 0

for mode in MODES_T4:
    fix_flag = (mode == 'fixed')
    for cfg in SIZE_CONFIGS:
        name    = cfg['name']
        D_GCN_v = cfg['D_GCN']
        D_GRU_v = cfg['D_GRU']
        for run_idx in range(NUM_RUNS_T4):
            ctr += 1
            key = (mode, name, run_idx)
            if key in done_t4:
                continue

            torch.manual_seed(run_idx)
            np.random.seed(run_idx)

            t0 = time.time()
            try:
                model = DenseGCNGRUPatched(
                    in_channels=2,
                    periods=T20,
                    batch_size=args.batch_size,
                    K=K_FIXED,
                    D_GCN=D_GCN_v,
                    D_GRU=D_GRU_v,
                    num_gru_layers=2,
                    fix_forward_bug=fix_flag,
                ).to(device)
                model, ckpt = train(
                    model, train_loader, val_loader, static_edge_index,
                    num_epochs=NUM_EPOCHS_T4, lr=args.lr,
                )
                model, ckpt = evaluate(
                    model, test_loader, static_edge_index, checkpoint_dict=ckpt,
                )
                mse = float(ckpt['test_mse'])
                mae = float(ckpt['test_mae'])
            except Exception as e:
                mse = float('nan')
                mae = float('nan')
                print(f'  !!! {mode} {name} run={run_idx} failed: {type(e).__name__}: {e}')
            finally:
                try:
                    del model
                except NameError:
                    pass
                torch.cuda.empty_cache()

            elapsed = time.time() - t0
            with table4_csv.open('a', newline='') as f:
                csv.writer(f).writerow(
                    [mode, name, D_GCN_v, D_GRU_v, K_FIXED, run_idx,
                     mse, mae, f'{elapsed:.1f}']
                )

            done_session += 1
            session_el = time.time() - ov_start
            avg = session_el / done_session
            eta = avg * (n_total_t4 - len(done_t4) - done_session)

            print(
                f'[{ctr:2d}/{n_total_t4}] mode={mode:14s} {name:9s} '
                f'(D_GCN={D_GCN_v}, D_GRU={D_GRU_v}) run={run_idx}  '
                f'MSE={mse:.4f}  MAE={mae:.4f}  '
                f'({elapsed:5.1f}s | ETA {eta/60:5.1f} min)'
            )

print(f"\nSweep complete. Session time: {(time.time()-ov_start)/60:.1f} min")
print(f"Results: {table4_csv.resolve()}")


### 6.3 Aggregate and compare to paper Table 4

Reported statistic: `mean ± (max − min) / 2` over 5 runs.


In [ ]:
df_t4 = pd.read_csv(table4_csv).dropna(subset=['test_mse', 'test_mae'])

agg_t4 = (df_t4
    .groupby(['mode', 'size_name', 'D_GCN', 'D_GRU'])
    .agg(
        mse_mean = ('test_mse', 'mean'),
        mse_hr   = ('test_mse', half_range),
        mae_mean = ('test_mae', 'mean'),
        mae_hr   = ('test_mae', half_range),
        n_runs   = ('test_mse', 'count'),
    )
    .reset_index()
)

# Preserve paper row order: Standard, Medium, Tiny
order = {'Standard': 0, 'Medium': 1, 'Tiny': 2}
agg_t4 = (agg_t4
    .assign(_ord=agg_t4['size_name'].map(order))
    .sort_values(['mode', '_ord'])
    .drop(columns='_ord')
    .reset_index(drop=True)
)

# Paper Table 4 reference numbers (from paper §5.5.2)
paper_t4 = pd.DataFrame([
    {'size_name': 'Standard', 'paper_mse': '0.0891 ± 0.0009', 'paper_mae': '0.2075 ± 0.0013'},
    {'size_name': 'Medium',   'paper_mse': '0.0949 ± 0.0013', 'paper_mae': '0.2126 ± 0.0032'},
    {'size_name': 'Tiny',     'paper_mse': '0.0987 ± 0.0004', 'paper_mae': '0.2156 ± 0.0025'},
])

print("=== Table 4 reproduction ===\n")
for mode in MODES_T4:
    sub = agg_t4[agg_t4['mode'] == mode]
    if len(sub) == 0:
        print(f"--- mode = {mode} : no runs yet ---\n")
        continue
    print(f"--- mode = {mode} ---")
    fmt = sub.merge(paper_t4, on='size_name')
    # Re-sort within merge result (merge can break order)
    fmt = (fmt
        .assign(_ord=fmt['size_name'].map(order))
        .sort_values('_ord')
        .drop(columns='_ord')
    )
    fmt['my_mse'] = fmt.apply(lambda r: f"{r['mse_mean']:.4f} ± {r['mse_hr']:.4f}", axis=1)
    fmt['my_mae'] = fmt.apply(lambda r: f"{r['mae_mean']:.4f} ± {r['mae_hr']:.4f}", axis=1)
    print(fmt[['size_name', 'D_GCN', 'D_GRU',
               'paper_mse', 'my_mse', 'paper_mae', 'my_mae', 'n_runs']]
          .to_string(index=False))
    print()


---

## Section 4–6 Summary

### What was added in this extension

| Section | Purpose | Output |
|---|---|---|
| 4.1 | Audit of `DenseGCNGRU` source — the K bug and the hardcoded hyperparameters | (markdown only) |
| 4.2 | `KLayerGCNConvPatched` and `DenseGCNGRUPatched` — exposes `K`, `D_GCN`, `D_GRU`, `num_gru_layers`, `fix_forward_bug` | code, in-notebook |
| 4.3 | Equivalence check: patched(default args, `fix_forward_bug=False`) ≡ original `DenseGCNGRU` | numerical assertion on `state_dict` and forward output |
| 5.x | Table 3 sweep across $K \in \{1,2,3\}$ × {paper_faithful, fixed} × 5 runs at $T=20$ | `table3_K_sensitivity.csv` |
| 6.x | Table 4 sweep across {Standard, Medium, Tiny} × {paper_faithful, fixed} × 5 runs at $T=20$ | `table4_size_sensitivity.csv` |

### What each mode is testing

| Mode | Table 3 ($K$ varies, $D_{GCN}$, $D_{GRU}$ fixed) | Table 4 ($D_{GCN}$, $D_{GRU}$ vary, $K=3$ fixed) |
|---|---|---|
| paper_faithful | Tests parameter-count effect of $K$ (extra GCN layers exist but never run). Should reproduce Dongdong's Table 3 numbers within seed variance. | Tests genuine $D_{GCN}$ / $D_{GRU}$ effect — the K bug applies but $K=3$ in all rows, so its effect is held constant. Should reproduce Dongdong's Table 4 numbers within seed variance. |
| fixed | Tests genuine $K$-hop receptive field. Numbers will differ from paper if the K-sensitivity claim was inflated by the bug. Not in the published paper. | Tests the size-reduction trade-off when $K=3$ is being respected. Informative for follow-up. Not in the published paper. |

### Internal consistency checks

- §4.3 — patched(defaults, fix=False) must produce a bit-identical `state_dict` and forward output to `DenseGCNGRU`. (asserted in-cell)
- §5.3 / §6.3 — Standard / paper_faithful in Table 4 should match $K=3$ / paper_faithful in Table 3, since both use $K=3$, $D_{GCN}=128$, $D_{GRU}=64$, no fix. (visual check across the two reports)

### What is *not* claimed by this section

- That the paper-faithful runs will reproduce Dongdong's reported numbers exactly. Seed differences and any unrecorded edits to `sten.py` between his Table 3 and Table 4 runs may cause drift; numbers within the published min-max range are the bar.
- That the `fix_forward_bug=True` results are what the paper *should* have reported. They are what the paper *would* have reported if the indentation in `KLayerGCNConv.forward` matched the prose. Whether the paper's K-sensitivity prose claim survives the fix is what this comparison answers.
- That `sten.py` itself has been modified. The patch lives only in this notebook (§4.2). The author's source file is untouched.
